# CIFAR-10: Large SE-ResNet teacher → legal student

**Experiment design:** Train a **15,904,586-parameter** SE-ResNet teacher *from scratch*, using only the allowed 45k training images. Fine-tune the original **4,215,242-parameter** student with and without knowledge distillation and compare on the same untouched 5k validation split. The teacher is an intermediate training model only.

**Run in stages.** In the code cell below, change `MODE` at the very top and execute the cell again:

1. **`teacher`** (default): 200-epoch teacher training. Recommend A100 if convenient; L4 is also workable. Saves `teacher_best.pth` and resumable state.
2. **`student_kd`**: 35-epoch fine-tune from your original `efficient_resnet_0878.pth`, with teacher soft targets. L4 is enough.
3. **`student_control`**: identical starting checkpoint, seeds, training recipe and schedule, without teacher loss. This is a *matched control* to isolate KD effects.
4. **`export`**: after reviewing **held-out validation only**, set `EXPORT_STUDENT` to the better legal student. Creates its `submission.csv`. No automatic Kaggle upload.

All checkpoints are under `MyDrive/dl-fa26-resnet-project/fast_runs/large_teacher_kd_v1/`; separate resume states prevent rerunning prior stages. **Do not use the teacher as a final inference model** (it exceeds the 5M cap).

**Caveat:** the fast GPU AutoAugment reproduces independent policy choices per image, but torchvision tensor operations can differ slightly from PIL. Set `AUGMENT_BACKEND='pil'` for slower exact PIL behavior if desired.


In [ ]:
"""Larger permitted teacher -> compliant SE-ResNet student, CIFAR-10 mini-project.

Colab standalone experiment; change MODE at top:
  teacher         : train 15.90M teacher from scratch on permitted 45k train, evaluate on permitted 5k val.
  student_kd      : fine-tune original legal 0.878 checkpoint using teacher knowledge.
  student_control : same training setup WITHOUT teacher KD (paired ablation).
  export          : create legal single-student submission.csv after choosing with validation.

The teacher is NEVER used at test inference; only the legal 4.215M student generates CSV.
No external datasets/weights are used. Re-running stage resumes its own state.
"""

# ======================= CHANGE ONLY THIS FIRST ========================
MODE = 'student_control'  # 'teacher', 'student_kd', 'student_control', 'export'
EXPORT_STUDENT = 'student_kd'  # 'student_kd' or 'student_control'; only for MODE='export'
# ======================================================================
RUN_NAME = 'large_teacher_kd_v1'
TEACHER_EPOCHS = 200
STUDENT_EPOCHS = 35
EPOCHS = TEACHER_EPOCHS if MODE == 'teacher' else STUDENT_EPOCHS
BATCH_SIZE = 256
SEED = 42
SAVE_EVERY = 5
RESUME = True
AUGMENT_BACKEND = 'gpu'  # fast independent CIFAR AutoAugment; 'pil' exact CPU PIL but much slower
BENCHMARK_AUGMENT = False
TEACHER_MAX_LR = 0.10
STUDENT_MAX_LR = 0.002
KD_TEMPERATURE = 3.0
KD_WEIGHT = 0.30
# New student fine-tunes the original legal checkpoint, NOT a third-party pretrained network.

import os, re, time, random, math, copy, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms.autoaugment import _apply_op

ROOT_DRIVE = Path('/content/drive/MyDrive/dl-fa26-resnet-project')
DATA_ZIP = ROOT_DRIVE / 'data.zip'
LOCAL_DATA = Path('/content/dataset')
RUN_DIR = ROOT_DRIVE / 'fast_runs' / RUN_NAME
BASELINE_PATH = ROOT_DRIVE / 'efficient_resnet_0878.pth'
TEACHER_BEST = RUN_DIR / 'teacher_best.pth'
MEAN_T = torch.tensor((0.4914, 0.4822, 0.4465), dtype=torch.float32).view(1, 3, 1, 1)
STD_T = torch.tensor((0.2023, 0.1994, 0.2010), dtype=torch.float32).view(1, 3, 1, 1)

def find_root(parent):
    if not parent.exists():
        return None
    for root, _, files in os.walk(parent):
        if 'train_labels.csv' in files and 'sample_submission.csv' in files:
            return Path(root)
    return None


def setup():
    from google.colab import drive
    drive.mount('/content/drive')
    if not torch.cuda.is_available():
        raise RuntimeError('Colab GPU not found. Runtime -> Change runtime type -> L4 / T4.')
    dev = torch.device('cuda')
    print('GPU:', torch.cuda.get_device_name(0), '| PyTorch:', torch.__version__)
    print('Run:', RUN_NAME, '| epochs:', EPOCHS, '| batch:', BATCH_SIZE)
    print('Augmentation:', AUGMENT_BACKEND, '| destination:', RUN_DIR)
    RUN_DIR.mkdir(exist_ok=True, parents=True)
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    np.random.seed(SEED)
    random.seed(SEED)
    root = find_root(LOCAL_DATA)
    if root is None:
        if not DATA_ZIP.exists():
            raise FileNotFoundError(f'Missing ZIP at: {DATA_ZIP}')
        LOCAL_DATA.mkdir(exist_ok=True, parents=True)
        print('Extracting permitted competition dataset...')
        with zipfile.ZipFile(DATA_ZIP) as z:
            z.extractall(LOCAL_DATA)
        root = find_root(LOCAL_DATA)
    if root is None:
        raise RuntimeError('Cannot locate train_labels.csv and sample_submission.csv.')
    print('Dataset:', root)
    return dev, root


# ---- Original model architecture and module names -----------------
# Same state-dict format as 0.878 teacher, widths legally increased.
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid())

    def forward(self, x):
        b, c, _, _ = x.shape
        return x * self.fc(self.avg_pool(x).view(b,c)).view(b,c,1,1)


class SEBasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.se = SEBlock(cout)
        self.shortcut = (nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),
                                       nn.BatchNorm2d(cout))
                         if cin != cout or stride != 1 else nn.Sequential())

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.se(self.bn2(self.conv2(out)))
        return F.relu(out + self.shortcut(x), inplace=True)


class SEResNet(nn.Module):
    def __init__(self, ch=(64,128,216,280)):
        super().__init__()
        self.conv1 = nn.Conv2d(3,ch[0],3,1,1,bias=False)
        self.bn1 = nn.BatchNorm2d(ch[0])
        self.layer1 = nn.Sequential(SEBasicBlock(ch[0],ch[0]),SEBasicBlock(ch[0],ch[0]))
        self.layer2 = nn.Sequential(SEBasicBlock(ch[0],ch[1],2),SEBasicBlock(ch[1],ch[1]))
        self.layer3 = nn.Sequential(SEBasicBlock(ch[1],ch[2],2),SEBasicBlock(ch[2],ch[2]))
        self.layer4 = nn.Sequential(SEBasicBlock(ch[2],ch[3],2),SEBasicBlock(ch[3],ch[3]))
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.linear = nn.Linear(ch[3],10)

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)), inplace=True)
        y = self.layer4(self.layer3(self.layer2(self.layer1(y))))
        return self.linear(self.avgpool(y).flatten(1))


def name_of(id_, test=False):
    s = str(id_).strip()
    if re.fullmatch(r'\d+\.0', s):
        s = s[:-2]
    n = ''.join(c for c in s if c.isdigit())
    if not n:
        raise ValueError(f'Bad image id: {id_!r}')
    return ('test_' + n.zfill(4) if test else 'train_' + n.zfill(5)) + '.png'


def load_pixels(df, folder, test=False):
    a = np.empty((len(df), 3, 32, 32), dtype=np.uint8)
    for i, id_ in enumerate(tqdm(df['id'], desc=('Test images' if test else 'Training images'))):
        p = folder / name_of(id_, test)
        with Image.open(p) as im:
            v = np.asarray(im.convert('RGB'), dtype=np.uint8)
        if v.shape != (32,32,3):
            raise ValueError(f'Invalid dimensions in {p}: {v.shape}')
        a[i] = v.transpose(2,0,1)
    return torch.from_numpy(a)


def normalize(batch, dev):
    return ((batch.float().div_(255.0) - MEAN_T.to(dev)) / STD_T.to(dev)).contiguous(memory_format=torch.channels_last)


# ---- Fast, independent per-image CIFAR10 AutoAugment ---------------
# Uses exactly the torchvision CIFAR10 *policies and magnitudes*. Each
# image draws its own policy, its own two Bernoulli trials and own signs.
# torchvision tensor image ops are used, which can differ slightly from PIL.
AUTO = transforms.AutoAugment(policy=transforms.AutoAugmentPolicy.CIFAR10,
                              interpolation=InterpolationMode.NEAREST, fill=0)
POLICIES = AUTO.policies
MAG_META = AUTO._augmentation_space(10, (32,32))


def random_crop_zero(x, pad=4):
    b, c, h, w = x.shape
    padded = F.pad(x, (pad, pad, pad, pad), mode='constant', value=0)
    dy = torch.randint(0, 2*pad+1, (b,), device=x.device)
    dx = torch.randint(0, 2*pad+1, (b,), device=x.device)
    ri = (dy[:,None] + torch.arange(h, device=x.device)[None,:])
    ri = ri[:,None,:,None].expand(b,c,h,w+2*pad)
    z = padded.gather(2, ri)
    ci = (dx[:,None] + torch.arange(w,device=x.device)[None,:])
    ci = ci[:,None,None,:].expand(b,c,h,w)
    return z.gather(3,ci)



def equalize_batch_uint8(x):
    """Vectorized torchvision/PIL-style histogram equalization, independently per image/channel.

    Faster on CUDA than torchvision's per-image Python loop.
    """
    b, c, h, w = x.shape
    flat = x.reshape(b*c, h*w).long()
    offsets = (torch.arange(b*c, device=x.device) * 256).unsqueeze(1)
    hist = torch.bincount((flat + offsets).reshape(-1), minlength=b*c*256).reshape(b*c, 256)
    last = torch.where(hist > 0, torch.arange(256,device=x.device), -1).amax(1)
    last_count = hist.gather(1, last[:,None])[:,0]
    step = torch.div(hist.sum(1) - last_count, 255, rounding_mode='floor')
    safe_step = step.clamp_min(1)
    # Match torchvision.transforms._functional_tensor._scale_channel exactly.
    lut = torch.div(hist.cumsum(1) + torch.div(safe_step[:,None],2,rounding_mode='floor'),
                    safe_step[:,None],rounding_mode='floor')
    lut = F.pad(lut, (1,0))[:,:-1].clamp(0,255)
    equalized = lut.gather(1,flat).reshape(b,c,h,w).to(torch.uint8)
    return torch.where((step > 0).reshape(b,c,1,1), equalized, x)

def autoaugment_gpu_by_image(x):
    # Input: a GPU uint8 tensor [N,3,32,32], 0-255.
    # All images have independent policy/op/sign decisions.
    b = x.shape[0]
    picks = np.random.randint(len(POLICIES), size=b)
    # Grouping policy IDs saves CPU->GPU overhead compared with PIL per image.
    for pidx in np.unique(picks):
        pos = np.flatnonzero(picks == pidx)
        z = x[pos].clone()
        for op_name, probability, magnitude_id in POLICIES[int(pidx)]:
            # Every image independently decides if op applies.
            apply_positions = np.flatnonzero(np.random.random(len(pos)) <= probability)
            if not len(apply_positions):
                continue
            levels, signed = MAG_META[op_name]
            mag = float(levels[magnitude_id].item()) if magnitude_id is not None else 0.0
            if signed:
                negative = np.random.random(len(apply_positions)) < .5
                signs_and_positions = ((-1, apply_positions[negative]),
                                       (+1, apply_positions[~negative]))
            else:
                signs_and_positions = ((+1, apply_positions),)
            for sign, sub_positions in signs_and_positions:
                if len(sub_positions):
                    chosen = z[sub_positions]
                    transformed = (equalize_batch_uint8(chosen) if op_name == 'Equalize'
                                   else _apply_op(chosen, op_name, mag*sign,
                                                  InterpolationMode.NEAREST, [0.,0.,0.]))
                    z[sub_positions] = transformed
        x[pos] = z
    return x


# Exact PIL baseline: only used when AUGMENT_BACKEND='pil', CPU bottleneck.
EXACT_PIL = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465), (0.2023,0.1994,0.2010))
])


@torch.no_grad()
def augment(x, dev):
    if AUGMENT_BACKEND == 'pil':
        # Use the same original PIL pipeline, independent per image.
        imgs = []
        for pix in x.cpu():
            im = Image.fromarray(pix.permute(1,2,0).numpy(), 'RGB')
            imgs.append(EXACT_PIL(im))
        return torch.stack(imgs).to(dev, non_blocking=True).contiguous(memory_format=torch.channels_last)
    elif AUGMENT_BACKEND == 'gpu':
        z = random_crop_zero(x, 4)
        flips = (torch.rand(len(z), device=dev) < .5).view(-1,1,1,1)
        z = torch.where(flips, z.flip(-1), z)
        z = autoaugment_gpu_by_image(z)
        return normalize(z, dev)
    raise ValueError('Unknown augmentation backend: ' + AUGMENT_BACKEND)


def cutmix(x, y):
    # Same original CutMix beta(alpha=1), same global batch cutout.
    lam = float(np.random.beta(1.0, 1.0))
    perm = torch.randperm(len(y), device=y.device)
    cut = math.sqrt(1.0 - lam)
    cw, ch = int(32 * cut), int(32 * cut)
    cx, cy = np.random.randint(32), np.random.randint(32)
    x1 = int(np.clip(cx-cw//2,0,32)); x2=int(np.clip(cx+cw//2,0,32))
    y1 = int(np.clip(cy-ch//2,0,32)); y2=int(np.clip(cy+ch//2,0,32))
    z = x.clone()
    z[:,:,y1:y2,x1:x2] = x[perm,:,y1:y2,x1:x2]
    lam_corrected = 1.0 - ((x2-x1)*(y2-y1)/(32*32))
    return z, y, y[perm], lam_corrected


@torch.inference_mode()
def evaluate(model, val_x, val_y, dtype, dev, corruption=None):
    model.eval()
    good_plain, good_flip, total = 0,0,0
    for start in range(0,len(val_y),BATCH_SIZE):
        end=min(start+BATCH_SIZE,len(val_y))
        raw = val_x[start:end].float().div_(255.)
        if corruption is not None:
            if corruption == 'brightness':
                raw = (raw * .72).clamp_(0, 1)
            elif corruption == 'contrast':
                mean = raw.mean(dim=(2,3), keepdim=True)
                raw = ((raw-mean)*.65+mean).clamp_(0,1)
            elif corruption == 'noise':
                # Reproducible noise for BOTH reference and new models.
                g = torch.Generator(device=dev).manual_seed(314159+start)
                raw = (raw + torch.randn(raw.shape, device=dev, generator=g)*.035).clamp_(0,1)
            elif corruption == 'blur':
                raw = F.avg_pool2d(raw,3,stride=1,padding=1)
            else:
                raise ValueError(corruption)
        x=((raw-MEAN_T.to(dev))/STD_T.to(dev)).contiguous(memory_format=torch.channels_last)
        y=val_y[start:end]
        with torch.autocast('cuda',dtype=dtype):
            a=model(x).float()
            b=model(x.flip(-1).contiguous(memory_format=torch.channels_last)).float()
        good_plain += (a.argmax(1)==y).sum().item()
        good_flip += (((a+b)*.5).argmax(1)==y).sum().item()
        total += len(y)
    return {'plain':100*good_plain/total, 'flip':100*good_flip/total}


def set_epoch_seed(epoch):
    # Ensures resumability across Colab session restarts at epoch boundaries.
    seed=SEED + epoch*971
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)





# Teacher is structurally different from the deployed student: deeper and larger.
# No teacher weights can be exported as the final legal model.
class TeacherSEResNet(nn.Module):
    def __init__(self, ch=(64, 128, 256, 384), blocks=(3, 4, 6, 3)):
        super().__init__()
        self.conv1 = nn.Conv2d(3, ch[0], 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(ch[0])
        self.layer1 = nn.Sequential(*[SEBasicBlock(ch[0], ch[0]) for _ in range(blocks[0])])
        for s in range(1, 4):
            layer = [SEBasicBlock(ch[s-1], ch[s], 2)]
            layer += [SEBasicBlock(ch[s], ch[s]) for _ in range(blocks[s]-1)]
            setattr(self, f'layer{s+1}', nn.Sequential(*layer))
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.linear = nn.Linear(ch[3], 10)

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)), inplace=True)
        y = self.layer4(self.layer3(self.layer2(self.layer1(y))))
        return self.linear(self.avgpool(y).flatten(1))


def make_student(dev):
    net = SEResNet((64,128,192,256)).to(dev).to(memory_format=torch.channels_last)
    n = sum(p.numel() for p in net.parameters() if p.requires_grad)
    assert n == 4_215_242 and n <= 5_000_000, n
    return net


def make_teacher(dev):
    net = TeacherSEResNet().to(dev).to(memory_format=torch.channels_last)
    n = sum(p.numel() for p in net.parameters() if p.requires_grad)
    assert n == 15_904_586, n
    return net


def split_data(root, dev):
    labels = pd.read_csv(root / 'train_labels.csv', dtype={'id':str})
    labels.columns = labels.columns.str.strip().str.lower()
    assert len(labels) == 50_000 and labels['id'].is_unique
    assert labels['label'].between(0,9).all()
    labels = labels.sample(frac=1,random_state=42).reset_index(drop=True)
    # Same reference split and pixel ordering used for our 0.878 experiment.
    pixels = load_pixels(labels,root/'train_images').to(dev)
    classes = torch.tensor(labels['label'].to_numpy(dtype=np.int64),device=dev)
    return pixels[:45000],classes[:45000],pixels[45000:],classes[45000:]


def load_weights(path, dev):
    obj=torch.load(path,map_location=dev,weights_only=True)
    return obj['model'] if isinstance(obj,dict) and 'model' in obj else obj


def save_weights(path, model):
    # state_dict includes trained BatchNorm running buffers, not just parameters.
    temp=path.with_suffix('.tmp')
    torch.save(model.state_dict(),temp)
    os.replace(temp,path)


def stage_paths(stage):
    return RUN_DIR/f'{stage}_best.pth', RUN_DIR/f'{stage}_resume.pt', RUN_DIR/f'{stage}_history.csv'


def mix_loss(logits, a,b,lam):
    return (lam * F.cross_entropy(logits,a,label_smoothing=.10) +
            (1-lam)*F.cross_entropy(logits,b,label_smoothing=.10))


@torch.no_grad()
def update_ema(ema, src, decay=.995):
    for e,p in zip(ema.parameters(),src.parameters()):
        e.lerp_(p, 1-decay)
    for e,p in zip(ema.buffers(),src.buffers()):
        e.copy_(p)


def train_stage(stage, xtr,ytr,xval,yval,dtype,dev):
    is_teacher = stage == 'teacher'
    is_kd = stage == 'student_kd'
    if stage not in {'teacher','student_kd','student_control'}:
        raise ValueError(stage)
    epochs=TEACHER_EPOCHS if is_teacher else STUDENT_EPOCHS
    max_lr=TEACHER_MAX_LR if is_teacher else STUDENT_MAX_LR
    model=make_teacher(dev) if is_teacher else make_student(dev)
    train_teacher=None
    if not is_teacher:
        if not BASELINE_PATH.exists():
            raise FileNotFoundError(f'Missing baseline student checkpoint: {BASELINE_PATH}')
        model.load_state_dict(load_weights(BASELINE_PATH,dev),strict=True)
        print('Student initialized from original legal 0.878 checkpoint')
        if is_kd:
            if not TEACHER_BEST.exists():
                raise FileNotFoundError(f'Train teacher first: {TEACHER_BEST}')
            train_teacher=make_teacher(dev)
            train_teacher.load_state_dict(load_weights(TEACHER_BEST,dev),strict=True)
            train_teacher.eval()
            train_teacher.requires_grad_(False)
            tv = evaluate(train_teacher,xval,yval,dtype,dev)
            print('Teacher used for KD:',tv)
    ema=copy.deepcopy(model).eval() if not is_teacher else None
    opt=torch.optim.SGD(model.parameters(),lr=max_lr,momentum=.9,weight_decay=5e-4)
    steps=math.ceil(len(ytr)/BATCH_SIZE)
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=max_lr,epochs=epochs,
        steps_per_epoch=steps,pct_start=(.30 if is_teacher else .10),div_factor=25,
        final_div_factor=1e4)
    scaler=torch.amp.GradScaler('cuda',enabled=(dtype == torch.float16))
    best_path,resume_path,hist_path=stage_paths(stage)
    signature={'stage':stage,'epochs':epochs,'batch':BATCH_SIZE,'max_lr':max_lr,
               'seed':SEED,'arch':('teacher_3463_64_128_256_384' if is_teacher else 'student_2222_64_128_192_256'),
               'augmentation':AUGMENT_BACKEND,'kd_weight':(KD_WEIGHT if is_kd else 0),
               'temperature':(KD_TEMPERATURE if is_kd else 0)}
    first=0;best=-1.;best_epoch=0;best_branch=None
    if RESUME and resume_path.exists():
        state=torch.load(resume_path,map_location=dev,weights_only=True)
        if state['signature']!=signature:
            raise ValueError('Configuration changed. Use a new RUN_NAME or matching settings to resume.')
        model.load_state_dict(state['model']);opt.load_state_dict(state['optimizer'])
        sched.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        if ema is not None: ema.load_state_dict(state['ema'])
        first=int(state['epoch']);best=float(state['best'])
        best_epoch=int(state['best_epoch']);best_branch=state['best_branch']
        print(f'Resuming {stage} from epoch {first+1}; best={best:.2f}%')
    else:
        print(f'Starting {stage}: {epochs} epochs; {sum(p.numel() for p in model.parameters()):,} parameters')
    started=time.perf_counter()
    for ep in range(first,epochs):
        set_epoch_seed(ep)  # paired student/control augment draws across epochs
        model.train()
        order=torch.randperm(len(ytr),device=dev)
        total_loss=torch.zeros((),device=dev)
        for start in range(0,len(ytr),BATCH_SIZE):
            idx=order[start:start+BATCH_SIZE]
            x=augment(xtr[idx],dev)
            y=ytr[idx]
            mix_p = .5 if is_teacher else (.25 if ep < int(epochs*.7) else 0.)
            if np.random.random() < mix_p:
                x,a,b,lam=cutmix(x,y)
            else:
                a=b=y;lam=1.
            teacher_logits=None
            if train_teacher is not None:
                with torch.no_grad(), torch.autocast('cuda',dtype=dtype):
                    teacher_logits=train_teacher(x).float()
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=dtype):
                logits=model(x)
                hard=mix_loss(logits,a,b,lam)
                if teacher_logits is not None:
                    temp=KD_TEMPERATURE
                    soft=F.kl_div(F.log_softmax(logits.float()/temp,dim=1),
                        F.softmax(teacher_logits/temp,dim=1),reduction='batchmean')*(temp**2)
                    loss=hard*(1-KD_WEIGHT)+soft*KD_WEIGHT
                else:
                    loss=hard
            if not bool(torch.isfinite(loss)):
                raise RuntimeError(f'Non-finite loss in {stage} at epoch {ep+1}')
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt);scaler.update();sched.step()
            if ema is not None:
                update_ema(ema,model,decay=.995)
            total_loss += loss.detach()
        raw=evaluate(model,xval,yval,dtype,dev)
        ema_val=evaluate(ema,xval,yval,dtype,dev) if ema is not None else None
        branches=[('raw',raw)] + ([('ema',ema_val)] if ema_val else [])
        for branch,val in branches:
            if val['flip'] > best:
                best=val['flip'];best_epoch=ep+1;best_branch=branch
                save_weights(best_path,model if branch=='raw' else ema)
        log={'epoch':ep+1,'loss':round(float(total_loss)/steps,4),
             'lr':round(float(sched.get_last_lr()[0]),7),'raw_flip':raw['flip'],
             'ema_flip':(ema_val['flip'] if ema_val else float('nan')),
             'best_flip':best,'best_epoch':best_epoch,'best_branch':best_branch}
        with hist_path.open('a') as f:
            if first==0 and ep==0:f.write(','.join(log.keys())+'\n')
            f.write(','.join(str(v) for v in log.values())+'\n')
        if is_teacher or (ep+1)%2==0 or ep==0 or ep==epochs-1:
            em_str='' if ema_val is None else f" | EMA={ema_val['flip']:.2f}%"
            print(f'{stage} {ep+1:3d}/{epochs} | loss={log["loss"]:.3f} | LR={log["lr"]:.5f}'
                  f' | raw={raw["flip"]:.2f}%{em_str} | BEST={best:.2f}%'
                  f' @ epoch {best_epoch} ({best_branch}) | {(time.perf_counter()-started)/60:.1f} min',flush=True)
        if (ep+1)%SAVE_EVERY==0 or ep==epochs-1:
            state={'signature':signature,'epoch':ep+1,'best':best,'best_epoch':best_epoch,
                   'best_branch':best_branch,'model':model.state_dict(),
                   'optimizer':opt.state_dict(),'scheduler':sched.state_dict(),'scaler':scaler.state_dict()}
            if ema is not None:state['ema']=ema.state_dict()
            tmp=resume_path.with_suffix('.tmp')
            torch.save(state,tmp)
            os.replace(tmp,resume_path)
    best_model=make_teacher(dev) if is_teacher else make_student(dev)
    best_model.load_state_dict(load_weights(best_path,dev))
    result=evaluate(best_model,xval,yval,dtype,dev)
    print(f'FINISHED {stage}: best {result} at epoch {best_epoch}, branch={best_branch}')
    print('Checkpoint:',best_path)
    return best_model,result


def report_stage(stage,xval,yval,dtype,dev):
    path,_,_=stage_paths(stage)
    if not path.exists():
        return None
    m=make_teacher(dev) if stage=='teacher' else make_student(dev)
    m.load_state_dict(load_weights(path,dev));m.eval()
    r={x:evaluate(m,xval,yval,dtype,dev, None if x=='clean' else x)
       for x in ('clean','brightness','contrast','noise','blur')}
    print(stage,'validation:',json.dumps(r,indent=2))
    (RUN_DIR/f'{stage}_evaluation.json').write_text(json.dumps(r,indent=2))
    del m
    return r


@torch.inference_mode()
def make_submission(model,test_pixels,sample,dtype,dev,path):
    model.eval()
    preds=[]
    for start in tqdm(range(0,len(test_pixels),BATCH_SIZE),desc='Test inference'):
        x=normalize(test_pixels[start:start+BATCH_SIZE],dev)
        with torch.autocast('cuda',dtype=dtype):
            z=model(x).float()
            zflip=model(x.flip(-1).contiguous(memory_format=torch.channels_last)).float()
        preds.extend(((z+zflip)*.5).argmax(1).cpu().tolist())
    out=sample.copy()
    assert out['id'].is_unique and len(out)==len(preds)
    out['label']=np.array(preds,dtype=np.int64)
    assert out['label'].between(0,9).all()
    out=out[['id','label']]
    out.to_csv(path,index=False)
    print('Saved compliant single-model CSV:',path)


def main():
    dev,root=setup()
    dtype=torch.bfloat16 if torch.cuda.get_device_capability(0)[0]>=8 else torch.float16
    print('AMP:',dtype)
    print('Teacher count:',sum(p.numel() for p in make_teacher(dev).parameters()))
    print('Student count:',sum(p.numel() for p in make_student(dev).parameters()))
    if MODE=='export':
        if EXPORT_STUDENT not in {'student_kd','student_control'}:
            raise ValueError('Only legal student models can be exported')
        path,_,_=stage_paths(EXPORT_STUDENT)
        if not path.exists():
            raise FileNotFoundError(f'First train {EXPORT_STUDENT}: {path}')
        net=make_student(dev)
        net.load_state_dict(load_weights(path,dev))
        sample=pd.read_csv(root/'sample_submission.csv',dtype={'id':str})
        sample.columns=sample.columns.str.strip().str.lower()
        assert list(sample.columns)==['id','label'] and sample['id'].is_unique
        test_x=load_pixels(sample,root/'test_images',test=True).to(dev)
        export_dir=RUN_DIR/'exports'/EXPORT_STUDENT
        export_dir.mkdir(parents=True,exist_ok=True)
        make_submission(net,test_x,sample,dtype,dev,export_dir/'submission.csv')
        print('Only the 4,215,242-parameter student generated predictions.')
        return
    xtr,ytr,xval,yval=split_data(root,dev)
    print('Permitted research split: training',len(ytr),'validation',len(yval))
    if MODE not in {'teacher','student_kd','student_control'}:
        raise ValueError(f'Unknown MODE={MODE}')
    if MODE != 'teacher' and not TEACHER_BEST.exists() and MODE=='student_kd':
        raise FileNotFoundError('Teacher must be trained first.')
    best,metrics=train_stage(MODE,xtr,ytr,xval,yval,dtype,dev)
    print('--- Held-out robustness probes ---')
    report_stage(MODE,xval,yval,dtype,dev)
    if BASELINE_PATH.exists():
        baseline=make_student(dev)
        baseline.load_state_dict(load_weights(BASELINE_PATH,dev))
        print('Original 0.878 checkpoint:',evaluate(baseline,xval,yval,dtype,dev))
        del baseline
    if MODE=='teacher':
        print('NEXT: If teacher looks competitive, change MODE to student_kd;'
              ' then run MODE=student_control with same conditions for a paired ablation.')
    else:
        print('Compare validation against 0.878 and other student stage BEFORE MODE=export.')
    print('No hidden-test labels or images were used in model selection.')


if __name__=='__main__':
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA L4 | PyTorch: 2.11.0+cu130
Run: large_teacher_kd_v1 | epochs: 35 | batch: 256
Augmentation: gpu | destination: /content/drive/MyDrive/dl-fa26-resnet-project/fast_runs/large_teacher_kd_v1
Dataset: /content/dataset
AMP: torch.bfloat16
Teacher count: 15904586
Student count: 4215242


Training images:   0%|          | 0/50000 [00:00<?, ?it/s]

Permitted research split: training 45000 validation 5000
Student initialized from original legal 0.878 checkpoint
Starting student_control: 35 epochs; 4,215,242 parameters
student_control   1/35 | loss=0.742 | LR=0.00044 | raw=97.10% | EMA=97.12% | BEST=97.12% @ epoch 1 (ema) | 0.2 min
student_control   2/35 | loss=0.736 | LR=0.00126 | raw=97.06% | EMA=97.04% | BEST=97.12% @ epoch 1 (ema) | 0.5 min
student_control   4/35 | loss=0.728 | LR=0.00200 | raw=96.84% | EMA=96.92% | BEST=97.12% @ epoch 1 (ema) | 0.9 min
student_control   6/35 | loss=0.700 | LR=0.00197 | raw=96.86% | EMA=97.00% | BEST=97.12% @ epoch 1 (ema) | 1.4 min
student_control   8/35 | loss=0.713 | LR=0.00190 | raw=97.10% | EMA=96.88% | BEST=97.12% @ epoch 1 (ema) | 1.9 min
student_control  10/35 | loss=0.707 | LR=0.00180 | raw=96.98% | EMA=97.00% | BEST=97.12% @ epoch 1 (ema) | 2.3 min
student_control  12/35 | loss=0.722 | LR=0.00166 | raw=96.90% | EMA=96.94% | BEST=97.12% @ epoch 1 (ema) | 2.8 min
student_control  14/35 

## Interpreting results

- **Teacher benchmark:** compare teacher flip-TTA accuracy to the untouched legal checkpoint (~97.04–97.10% on this split). A weaker teacher may offer little advantage.
- **Causal ablation:** compare `student_kd_best.pth` vs `student_control_best.pth`. Both begin from the same legal checkpoint and use the same seeds and augmentation. Small differences on 5,000 images can still be noise.
- **Export rule:** only the selected **4.215M student** is run on the 2,000-image hidden set to produce a CSV; hidden images are not used in this notebook for model selection. The official Kaggle process is the only hidden-set feedback source.
- **Future final run:** after recipe selection, consider training the final student on all 50k allowed images. Do not call the former 5k subset *validation* once it has entered training.
